In [18]:
!wget https://github.com/rhasspy/piper/releases/download/v1.2.0/piper_amd64.tar.gz

--2026-06-28 10:47:44--  https://github.com/rhasspy/piper/releases/download/v1.2.0/piper_amd64.tar.gz
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/587499842/873099af-72dc-461a-bb2b-670eca1ebd37?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-06-28T11%3A32%3A56Z&rscd=attachment%3B+filename%3Dpiper_amd64.tar.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-06-28T10%3A32%3A11Z&ske=2026-06-28T11%3A32%3A56Z&sks=b&skv=2018-11-09&sig=c5%2FUp3TSsKNlYJp14aNizOQQe1YKlYK3J17CfEP4Mf8%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4MjY0NTQ2NCwibmJmIjoxNzgyNjQzNjY0LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5i

In [40]:
!wget -O en_US-lessac-medium.onnx \
https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/en_US-lessac-medium.onnx

!wget -O en_US-lessac-medium.onnx.json \
https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/en_US-lessac-medium.onnx.json

--2026-06-28 10:55:25--  https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/en_US-lessac-medium.onnx
Resolving huggingface.co (huggingface.co)... 3.171.171.65, 3.171.171.128, 3.171.171.104, ...
Connecting to huggingface.co (huggingface.co)|3.171.171.65|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/64949e47c841c90374a1fb82/514e1d7ed3038a07eec8f59c3f78739fb15eb7831cd295f44f36f3ba96b26183?user_id=public&X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27en_US-lessac-medium.onnx%3B+filename%3D%22en_US-lessac-medium.onnx%22%3B&Expires=1782647725&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjQ5NDllNDdjODQxYzkwMzc0YTFmYjgyLzUxNGUxZDdlZDMwMzhhMDdlZWM4ZjU5YzNmNzg3MzlmYjE1ZWI3ODMxY2QyOTVmNDRmMzZmM2JhOTZiMjYxODNcXD91c2VyX2lkPXB1YmxpYyZYLVhldC1DYXMtVWlkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPWlubGluZSUzQitma

In [19]:
!tar -xzf piper_amd64.tar.gz

In [2]:
!pip install faster-whisper
!pip install sounddevice
!pip install scipy
!pip install numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 62.9 MB/s eta 0:00:00


In [5]:
!pip install faster-whisper
!apt-get install ffmpeg -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [43]:
from IPython.display import Javascript
from google.colab import output
from base64 import b64decode

RECORD = """
async function recordAudio() {
  const stream = await navigator.mediaDevices.getUserMedia({audio:true});
  const recorder = new MediaRecorder(stream);
  const chunks = [];

  recorder.ondataavailable = e => chunks.push(e.data);

  recorder.start();

  await new Promise(resolve => setTimeout(resolve, 5000));

  recorder.stop();

  await new Promise(resolve => recorder.onstop = resolve);

  const blob = new Blob(chunks);
  const reader = new FileReader();

  reader.readAsDataURL(blob);

  return await new Promise(resolve=>{
      reader.onloadend=()=>resolve(reader.result);
  });
}
"""

display(Javascript(RECORD))

data = output.eval_js("recordAudio()")

binary = b64decode(data.split(",")[1])

with open("audio.webm","wb") as f:
    f.write(binary)

print("Audio saved.")

<IPython.core.display.Javascript object>

Audio saved.


In [44]:
!ffmpeg -i audio.webm audio.wav -y

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [45]:
from faster_whisper import WhisperModel

model = WhisperModel("base", device="cpu")

segments, info = model.transcribe("audio.wav")

question = ""

for segment in segments:
    question += segment.text

print("Question:")
print(question)

Question:
 What are the symptoms of diabetes?


In [18]:
!pip install tf-keras

In [19]:
!pip install transformers==4.52.4 tensorflow==2.19.0 torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 61.4 MB/s eta 0:00:00
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.20.0
    Uninstalling tensorboard-2.20.0:
      Successfully uninstalled tensorboard-2.20.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.19.0
    Uninstalling huggingface_hub-1.19.0:
      Successfully uninstalled huggingface_hub-1.19.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation:

In [1]:
from transformers import AutoTokenizer, TFAutoModelForQuestionAnswering
import tensorflow as tf

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-cased-distilled-squad"
)

qa_model = TFAutoModelForQuestionAnswering.from_pretrained(
    "distilbert-base-cased-distilled-squad"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

All PyTorch model weights were used when initializing TFDistilBertForQuestionAnswering.

All the weights of TFDistilBertForQuestionAnswering were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFDistilBertForQuestionAnswering for predictions without further training.


In [2]:
context = """
Diabetes is a chronic disease that occurs when blood glucose levels become too high.

Symptoms include increased thirst,
frequent urination,
fatigue,
blurred vision,
and weight loss.

Treatment includes healthy eating,
regular exercise,
insulin therapy,
and oral medications prescribed by a doctor.
"""

In [46]:
inputs = tokenizer(
    question,
    context,
    return_tensors="tf",
    truncation=True
)

outputs = qa_model(**inputs)

start = tf.argmax(outputs.start_logits, axis=1).numpy()[0]
end = tf.argmax(outputs.end_logits, axis=1).numpy()[0] + 1

tokens = inputs["input_ids"][0][start:end]

answer = tokenizer.decode(tokens)

print("\nQuestion:")
print(question)

print("\nAnswer:")
print(answer)


Question:
 What are the symptoms of diabetes?

Answer:
increased thirst, frequent urination, fatigue, blurred vision, and weight loss


In [47]:
answer = answer.replace("[CLS]", "").replace("[SEP]", "")

In [48]:
with open("answer.txt","w") as f:
    f.write(answer)

In [49]:
!echo "$answer" | ./piper/piper \
    --model en_US-lessac-medium.onnx \
    --output_file answer.wav

[2026-06-28 10:56:37.673] [piper] [info] Loaded voice in 0.366710604 second(s)
[2026-06-28 10:56:37.677] [piper] [info] Initialized piper
answer.wav
[2026-06-28 10:56:38.625] [piper] [info] Real-time factor: 0.1775717879921361 (infer=0.940091621 sec, audio=5.294149659863946 sec)
[2026-06-28 10:56:38.625] [piper] [info] Terminated piper


In [50]:
from IPython.display import Audio

Audio("answer.wav")